In [ ]:
import os
import datetime
import colormaps
from pathlib import Path
from functools import partial
from itertools import product
from string import ascii_lowercase
from tqdm import tqdm, trange
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
from matplotlib.colors import (
    LinearSegmentedColormap,
    BoundaryNorm,
    Normalize,
    rgb_to_hsv,
    hsv_to_rgb,
)
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
import xarray as xr
import polars as pl
import polars.selectors as cs

os.environ["PATH"] = os.environ["PATH"] + ":/opt/homebrew/bin"

from jetutils.definitions import (
    PRETTIER_VARNAME,
    YEARS,
    UNITS,
    RESULTS,
    FACTORS_UNITS,
    FACTORS,
    DATADIR,
    DUNCANS_REGIONS_NAMES,
    MONTH_NAMES,
    FIGURES,
    RADIUS,
    get_region,
    squarify,
    polars_to_xarray,
    xarray_to_polars,
    compute,
)
from jetutils.data import (
    standardize,
    standardize_polars_dtypes,
    compute_anomalies_pl,
    DataHandler,
    open_da,
)
from jetutils.geospatial import (
    central_diff,
    haversine_from_dl,
    compute_relative_clim,
    compute_relative_sm,
    compute_relative_std,
    compute_relative_anom,
    create_all_relative_plots,
)
from jetutils.jet_finding import (
    average_jet_categories,
    track_jets,
    slowness_from_cross,
    spells_from_cross,
    spells_from_cross_catd_simple,
    extract_features,
    find_all_jets,
    jet_position_as_da,
    to_one_large,
    gaussian_smooth_func
)
from jetutils.plots import (
    STYLE_SHEET,
    COLORS,
    COLORS_EXT,
    WERNLI_FLAIR,
    WERNLI_FLAIR_LEVELS,
    Clusterplot,
    plot_interp,
    plot_relative_time,
    gaussian_kde,
)
from jetutils.anyspell import (
    make_daily,
    mask_from_spells_pl,
    subset_around_onset,
    extend_spells,
    get_spells,
)
from jetutils.stats import (
    create_bootstrapped_times,
    odds_ratio,
    is_signif_OR,
    common_OR,
)

%load_ext IPython.extensions.autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

basepath = Path(f"{DATADIR}/ERA5/plev/high_wind/6H/results/9")


ALL_TIMES = (
    pl.datetime_range(
        start=pl.datetime(1959, 1, 1),
        end=pl.datetime(2023, 1, 1),
        closed="left",
        interval="6h",
        eager=True,
        time_unit="ms",
    )
    .rename("time")
    .to_frame()
)
summer_filter = ALL_TIMES.filter(pl.col("time").dt.month().is_in([7, 8, 9]))
summer = summer_filter["time"]
summer_daily = summer.filter(summer.dt.hour() == 0)

In [ ]:
jets = pl.read_parquet(basepath.joinpath("jets.parquet"))
phat_jets = to_one_large(jets)
props = pl.read_parquet(basepath.joinpath("props.parquet"))
cross = pl.read_parquet(basepath.joinpath("cross.parquet"))
pers = slowness_from_cross(cross).drop("is_polar")
props = props.join(pers, on=["time", "jet ID"])

over_europe = pl.col("lon") > -10
lat_over_europe = (pl.col("lat") * pl.col("s")).filter(over_europe).sum() / pl.col(
    "s"
).filter(over_europe).sum()
lat_over_europe = jets.group_by("time", "jet ID").agg(
    lat_over_europe.fill_nan(0).alias("lat_over_europe")
)
props = props.join(lat_over_europe, on=["time", "jet ID"])

props_catd = squarify(average_jet_categories(props), ["time", "jet"])
props_catd = props_catd.join(
    props_catd.rolling("time", period="2d", group_by="jet").agg(
        **{
            f"{col}_var": pl.col(col).var()
            for col in ["mean_lon", "mean_lat", "mean_s", "s_star"]
        }
    ),
    on=["time", "jet"],
)
pers = props_catd.rolling("time", period="5d", group_by="jet").agg(pers=pl.col("slowness").sum())
props_catd = props_catd.join(pers, on=("time", "jet"))

cross_catd_ofile = basepath.joinpath("cross_catd.parquet")
if cross_catd_ofile.is_file():
    cross_catd = pl.read_parquet(cross_catd_ofile)
else:
    cross_catd = track_jets(phat_jets)
    cross_catd.write_parquet(cross_catd_ofile)
slowness_catd = slowness_from_cross(cross_catd).rename({"slowness": "slowness_catd"})
pers_catd = slowness_catd.rolling("time", period="5d", group_by="jet").agg(pers_catd=pl.col("slowness_catd").sum())
pers_catd = slowness_catd.join(pers_catd, on=("time", "jet"))
phat_filter = (pl.col("is_polar") < 0.5) | (
    (pl.col("is_polar") > 0.5) & (pl.col("int") > 1.3e8)
)
phat_props = props.filter(phat_filter)
phat_props = squarify(average_jet_categories(phat_props), ["time", "jet"])
phat_props = phat_props.join(
    phat_props.rolling("time", period="2d", group_by="jet").agg(
        **{
            f"{col}_var": pl.col(col).var()
            for col in ["mean_lon", "mean_lat", "mean_s", "s_star"]
        }
    ),
    on=["time", "jet"],
)
phat_props = phat_props.join(pers_catd.drop("is_polar"), on=("time", "jet"))
phat_props_summer = summer_filter.join(phat_props, on="time")
props_catd_summer = summer_filter.join(props_catd, on="time")


spells_list = spells_from_cross(
    jets,
    cross,
    2.2e6,
    0.4,
    0.12,
    n_STJ=64,
    n_EDJ=30,
    season=summer,
)
filtering_for_STJ = props_catd.select("time", "jet", "mean_lat", "mean_s").pivot(on="jet", index="time").drop(cs.contains("EDJ"))
filtering_for_STJ = spells_list["STJ"].join(filtering_for_STJ, on="time").group_by("spell").agg(pl.col("mean_lat_STJ").mean())
filtering_for_STJ = filtering_for_STJ.filter(pl.col("mean_lat_STJ") > 33) 
spells_list["STJ"] = filtering_for_STJ.select("spell").join(spells_list["STJ"], on="spell")
spells_list["STJ"] = spells_list["STJ"].with_columns(pl.col("spell").rle_id()).sort("spell", "time")
spells_list["EDJ"] = spells_list["EDJ"].sort("spell", "time")
for jet, spells in spells_list.items():
    print(jet, spells["spell"].n_unique())
    print(jet, spells["len"].min() / 4)

daily_spells_list = {
    a: make_daily(b, "spell", ["len", "spell_of"]) for a, b in spells_list.items()
}

ds = xr.open_dataset(basepath.joinpath("da.nc"))

# grams weather regimes

In [ ]:
dh = DataHandler.from_specs(
    "ERA5",
    "plev",
    "z",
    "6H",
    "all",
    [6, 7, 8, 9],
    -80,
    40,
    15,
    80,
    500,
    "dayofyear",
    {"dayofyear": ("win", 15)},
    None,
)

exp_z = Experiment(dh)
centers_z, labels_z = exp_z.do_kmeans(4, 30, weigh_grams=False)

coslat = np.cos(np.deg2rad(centers_z.lat))
Pwr = (
    (dh.da * centers_z * coslat).sum(["lon", "lat"])
    / coslat.sum()
    / centers_z.lon.shape[0]
)
Iwr = (Pwr - Pwr.mean("time")) / Pwr.std("time", ddof=0)
Iwr = compute(Iwr)
Iwr = xarray_to_polars(Iwr.rename("Iwr")).drop("ratio", "label")

# Iwr = Iwr.with_columns(year=pl.col("time").dt.year())
sigma_wr = Iwr["Iwr"].std()
winner = Iwr.group_by("time", maintain_order=True).agg(
    pl.col("Iwr").arg_max().alias("winner"),
    pl.col("Iwr").max(),
    pl.col("time").dt.year().alias("year").first(),
)
winner = winner.with_columns(
    winner=pl.when(pl.col("Iwr") > sigma_wr)
    .then(1 + pl.col("winner"))
    .otherwise(pl.lit(0))
)
start_of_year = (
    winner.group_by("year", maintain_order=True)
    .len()
    .with_columns(start_of_year=pl.col("len").cum_sum() - pl.col("len").get(0))
    .drop("len")
)
to_zero = (
    winner.group_by("year", maintain_order=True)
    .agg(pl.col("winner").rle().alias("rle"))
    .explode("rle")
    .unnest("rle")
    .group_by("year", maintain_order=True)
    .agg(
        len=pl.col("len"),
        start=pl.lit(0).append(
            pl.col("len").cum_sum().slice(0, pl.col("len").len() - 1)
        ),
        value=pl.col("value"),
    )
    .explode(["len", "start", "value"])
    .join(start_of_year, on="year")
    .with_columns(start=pl.col("start") + pl.col("start_of_year"))
    .drop("start_of_year")
    .filter(pl.col("len") < 20, pl.col("value") > 0)
    .drop("value")
    .with_columns(index=pl.int_ranges(pl.col("start"), pl.col("start") + pl.col("len")))
    .drop("len", "start")
    .explode("index")
)[:, "index"]
winner[to_zero, "winner"] = 0

mask_updated = labels_to_mask(winner[:, "winner"].to_numpy())

In [ ]:
["Greenland Blocking", "Zonal regime", "EuBL/AR", "ScBl"]

In [ ]:
clu = Clusterplot(2, 2, get_region(exp_z.da))
clu.add_contourf(
    [dh.da[winner["winner"] == i].mean("time") / 9.8 for i in range(1, 5)],
    cmap=colormaps.BlWhRe,
    levels=[-100, -60, -20, 20, 60, 100],
    titles=[str(i) for i in range(1, 5)],
)
a = 1

In [ ]:
winner.write_parquet(dh.path.joinpath("grams_wr.parquet"))

# Real space composites

## data over europe

In [ ]:
args1 = ["all", [6, 7, 8, 9], -10, 40, 30, 70]
args2 = ["dayofyear", {"dayofyear": ("win", 15)}]

da_T_anom_europe = DataHandler.from_specs(
    "ERA5",
    "surf",
    "t2m",
    "6H",
    *args1,
    "all",
    *args2,
).da
da_T_anom_europe = compute(da_T_anom_europe, progress_flag=True)
da_tp_anom_europe = DataHandler.from_specs(
    "ERA5",
    "surf",
    "tp",
    "6H",
    *args1,
    "all",
    *args2,
).da
da_tp_anom_europe = compute(da_tp_anom_europe, progress_flag=True)
ds_ = compute(ds.isel(time=np.isin(ds.time.dt.month, [6, 7, 8, 9])).chunk("auto"), progress_flag=True)

## data over Natl

In [ ]:
args1 = ["all", [6, 7, 8, 9], *get_region(ds)]
args2 = ["dayofyear", {"dayofyear": ("win", 15)}]

# da_tp = DataHandler.from_specs(
#     "ERA5", "surf", "tp", "6H", "all", [6, 7, 8, 9], *(-80, 40, 0, 90)
# ).da
# da_tp = compute(da_tp, progress_flag=True)
da_tp_anom = DataHandler.from_specs(
    "ERA5", "surf", "tp", "6H", "all", [6, 7, 8, 9], *(-80, 40, 0, 90), "all",
    *args2,
).da
da_tp_anom = compute(da_tp_anom, progress_flag=True)
da_theta2pvu = DataHandler.from_specs(
    "ERA5", "surf", ("alot2pvu", "theta"), "6H", *args1
).da
da_theta2pvu = compute(da_theta2pvu, progress_flag=True)
da_T_anom = DataHandler.from_specs(
    "ERA5",
    "surf",
    "t2m",
    "6H",
    "all",
    [6, 7, 8, 9],
    *(-80, 40, 0, 90),
    "all",
    *args2,
).da
da_T_anom = compute(da_T_anom, progress_flag=True)
da_z500_anom = DataHandler.from_specs(
    "ERA5",
    "plev",
    "z",
    "6H",
    *args1,
    500,
    *args2,
).da
da_z500_anom = compute(da_z500_anom, progress_flag=False).rename("z500") / 9.8
varnames_rwb = ["APVO_new", "CPVO_new"]
das_rwb = {}
for var in varnames_rwb:
    das_rwb[var] = compute(
        DataHandler.from_specs(
            "ERA5", "thetalev", var, "6H", *args1, reduce_da=False
        ).da,
        progress_flag=True,
    )
# ds_ = compute(ds.isel(time=np.isin(ds.time.dt.month, [6, 7, 8, 9])).chunk("auto"), progress_flag=True)

In [ ]:
da_tp = DataHandler.from_specs(
    "ERA5", "surf", "tp", "6H", "all", [6, 7, 8, 9], *(-80, 40, 0, 90)
).da
da_tp = compute(da_tp, progress_flag=True)

In [ ]:
da_tp[5000].plot()

In [ ]:
import cdsapi

dataset = "reanalysis-era5-single-levels"
request = {
    "product_type": ["reanalysis"],
    "variable": ["total_precipitation"],
    "year": [f"{year}" for year in range(1959, 2023)],
    "month": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12"
    ],
    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ],
    "time": [
        "00:00", "06:00", "12:00",
        "18:00"
    ],
    "data_format": "grib",
    "download_format": "unarchived",
    "area": [90, -80, 0, 40]
}

client = cdsapi.Client()
client.retrieve(dataset, request).download(f"{DATADIR}/ERA5/precip.grib")

In [ ]:
xr.open_dataset(f"{DATADIR}/ERA5/precip.grib", engine="cfgrib")

In [ ]:
xr.open_dataarray("/storage/workspaces/giub_meteo_impacts/ci01/ERA5/surf/tp/6H/1959.nc")

In [ ]:
da_tp[:, :20, :].mean(["lon", "lat"]).plot()

## create timeseries

In [ ]:
from jetutils.data import extract
import matplotlib.patches as mpatches
import cartopy.crs as ccrs

clu = Clusterplot(1, 1, (-85, 55, 0, 90), row_height=6)
ax = clu.axes[0]
gl = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.4, color='k', linestyle='--')

boxes_precip = {"tropical": (-80, 40, 0, 15), "gulfstream": (-80, -50, 20, 45)}
boxes_temp = {"tropical": (-80, 40, 0, 15), "west": (-80, -40, 20, 60), "east": (-40, -0, 20, 60)}

for name, box in boxes_precip.items():
    ax.add_patch(
        mpatches.Rectangle(
            xy=[box[0], box[2]],
            width=box[1] - box[0],
            height=box[3] - box[2],
            facecolor="none",
            edgecolor="cyan",
            linewidth=2,
            transform=ccrs.PlateCarree(),
        )
    )
    huh = extract(da_tp, "all", None, *box).mean(["lon", "lat"])
    opath = Path(DATADIR, "ERA5/timeseries/", f"{huh.name}_{name}.parquet")
    huh = xarray_to_polars(huh)
    huh.write_parquet(opath)
for name, box in boxes_temp.items():
    ax.add_patch(
        mpatches.Rectangle(
            xy=[box[0], box[2]],
            width=box[1] - box[0],
            height=box[3] - box[2],
            facecolor="none",
            edgecolor="red",
            linestyle="dashed",
            linewidth=2,
            transform=ccrs.PlateCarree(),
        )
    )
    huh = extract(da_T_anom, "all", None, *box).mean(["lon", "lat"])
    opath = Path(DATADIR, "ERA5/timeseries/", f"{huh.name}_{name}.parquet")
    huh = xarray_to_polars(huh)
    huh.write_parquet(opath)

In [ ]:
# from jetutils.anyspell import regionalize
# region_T_ts = regionalize(da_T_anom, clusters_da, ["time"])
# region_tp_ts = regionalize(da_tp_anom, clusters_da, ["time"])

In [ ]:
# region_T_ts.write_parquet("/storage/workspaces/giub_meteo_impacts/ci01/ERA5/surf/t2m/dailymean/dayofyear_doywin15/results/1/regionalized.parquet")

In [ ]:
# region_tp_ts.write_parquet("/storage/workspaces/giub_meteo_impacts/ci01/ERA5/surf/tp/dailysum/dayofyear_doywin15/results/1/regionalized.parquet")

## Atlantic composites

In [ ]:
from jetutils.stats import field_significance_for_spells

to_plot = {}
pvals = {}
fdr_signif = {}

for da, jet in product([da_T_anom, da_tp_anom, da_z500_anom], ["STJ", "EDJ"]):
    key = f"{da.name}_{jet}"
    to_plot[key], pvals[key], fdr_signif[key] = field_significance_for_spells(da, spells_list[jet], summer, n_bootstraps=50, q=0.05)

In [ ]:
xr.Dataset(to_plot).to_netcdf(basepath.joinpath("NAtl.nc"))
xr.Dataset(pvals).to_netcdf(basepath.joinpath("NAtl_pvals.nc"))
xr.Dataset(fdr_signif).to_netcdf(basepath.joinpath("NAtl_fdr.nc"))

## Europe composites

#### just plot data

In [ ]:
from jetutils.stats import field_significance

reduction_function = np.nanmean
plt.rc("axes", titlesize=14)
da_contour = da_z500_anom.sel(time=da_z500_anom.time.dt.year >= 1959)
da_1 = da_T_anom_europe
da_2 = da_tp_anom_europe * 1000
days_around = 3
mask = np.zeros((len(da_contour.time), len(spells_list)), dtype=bool)
time_name = "time"
for j, jet in enumerate(spells_list):
    spells_from_jet = subset_around_onset(
        spells_list[jet], around_onset=datetime.timedelta(days=days_around)
    )
    mask[:, j] = np.isin(da_contour.time.values, spells_from_jet["time"])
    n_spells = spells_from_jet["spell"].n_unique()

to_plot = {}
signifs = {}
for da in [da_1, da_2]:
    to_plot_cf = []
    for mas in tqdm(mask.T, total=mask.shape[1]):
        if np.sum(mas) < 1:
            to_plot_cf.append(da[0].copy(data=np.zeros(da.shape[1:])))
            continue
        to_plot_cf.append(
            da.isel({time_name: mas}).reduce(reduction_function, dim=time_name)
        )
    to_plot[da.name] = xr.concat(to_plot_cf, dim="jet")

    to_test = []
    for mas in mask.T:
        if np.sum(mas) < 1:
            to_test.append(da[:1].copy(data=np.zeros((1, *da.shape[1:]))))
            continue
        to_test.append(da.isel(time=mas))
    significances = []
    da_ = da.values
    # da = np.sort(da, axis=0)
    for i in trange(mask.shape[1]):
        this_signif = field_significance(to_test[i], da_, 200, q=0.1)[1]
        this_signif = da[0].copy(data=this_signif).reset_coords("time", drop=True)
        significances.append(this_signif)
    signifs[da.name] = xr.concat(significances, dim="jet")

to_plot = xr.Dataset(to_plot)
signifs = xr.Dataset(signifs)

to_plot_c = []
for mas in tqdm(mask.T, total=mask.shape[1]):
    if np.sum(mas) < 1:
        to_plot_c.append(da_contour[0].copy(data=np.zeros(da.shape[1:])))
        continue
    to_plot_c.append(
        da_contour.isel({time_name: mas}).reduce(reduction_function, dim=time_name)
    )
to_plot_c = xr.concat(to_plot_c, dim="jet")

masked_da = []
time_name = "time"
reduction_function = np.nanmean
for mas in mask.T:
    masked_da.append(
        ds_.isel({time_name: mas}).reduce(reduction_function, dim=time_name)
    )
masked_da = xr.concat(masked_da, dim="relative_index")
find_jets_kwargs = dict(
    n_coarsen=1,
    base_s_thresh=20,
    alignment_thresh=0.6,
    int_thresh_factor=0.6,
    hole_size=6,
    smooth_func=partial(gaussian_smooth_func, sigma_lon=2, sigma_lat=0.8),
)
jets_on_mean = find_all_jets(masked_da, **find_jets_kwargs)

to_plot.to_netcdf(
    "/storage/homefs/hb22g102/jetutils/Results/figure_data_pers/contourf.nc"
)
signifs.to_netcdf(
    "/storage/homefs/hb22g102/jetutils/Results/figure_data_pers/signifs.nc"
)
to_plot_c.to_netcdf(
    "/storage/homefs/hb22g102/jetutils/Results/figure_data_pers/contour.nc"
)
masked_da.to_netcdf(
    "/storage/homefs/hb22g102/jetutils/Results/figure_data_pers/wind.nc"
)

## Individual spells

#### the case study plot

In [ ]:
from jetutils.data import compute_anomalies_pl, coarsen_da

n_stj = 23
n_edj = 9
figure = plt.figure(layout="constrained", figsize=(9, 5.6))
subfigs = figure.subfigures(1, 2, wspace=0.01)
fig1, fig2 = subfigs
nrow, ncol = 3, 2
n = nrow * ncol
plt.rc("axes", titlesize=14)
plt.rc("axes", titlepad=2)

rwb_color = {
    "S": "mediumseagreen",
    "T": "orangered",
}
rwb_hatching = {
    "A": r"xx",
    "C": r"++",
}
rwb_lw = 2

cbar_kwargs = {"location": "bottom", "shrink": 0.8, "fraction": 0.1, "pad": 0.03}
plot_kwargs_all = {
    "STJ": {
        "cmap": colormaps.cmp_b2r,
        "levels": MaxNLocator(7).tick_values(-10, 10).tolist(),
        "cbar_kwargs": cbar_kwargs,
    },
    "EDJ": {
        "cmap": colormaps.bilbao_r,
        "levels": MaxNLocator(8).tick_values(300, 390).tolist(),
        "cbar_kwargs": cbar_kwargs,
    },
}

for fig, key, da, which_spell in zip(
    subfigs, ["STJ", "EDJ"], [da_T_anom, da_theta2pvu], [n_stj, n_edj]
):
    is_edj = key == "EDJ"
    plot_kwargs = plot_kwargs_all[key]
    region = (-80, 40, 15, 80) if key == "EDJ" else (-80, 40, 0, 65)
    clu = Clusterplot(nrow, ncol, region, row_height=3.5, fig=fig)
    spell = spells_list[key].filter(pl.col("spell") == which_spell)
    spell = extend_spells(spell, time_before=datetime.timedelta(days=0))
    len_spell = spell["time"][-1] - spell["time"][0]

    times = (
        pl.linear_space(spell["time"][0], spell["time"][-1], n, eager=True)
        .dt.round("6h")
        .cast(pl.Datetime("ms"))
    )
    titles = times.dt.to_string("%Y-%m-%dT%H:00").to_list()
    titles = [
        f"{ascii_lowercase[n * is_edj + i]}) {title}" for i, title in enumerate(titles)
    ]
    to_plot = da.sel(time=times)
    clu.add_contourf(
        to_plot,
        titles=titles,
        cbar_label=PRETTIER_VARNAME.get(da.name, da.name),
        **plot_kwargs,
    )
    fig.suptitle(f"{key} episode, " + str(len_spell)[:6])

    for i, t in enumerate(times):
        if i == 0 and t < spell["time"][0]:
            t_ = spell["time"][0]
        else:
            t_ = t
        jets_ = jets.filter(pl.col("time") == t_)
        for _, jet in jets_.group_by("jet ID"):
            lo, la, is_p = jet[["lon", "lat", "is_polar"]]
            this_jet_is_p = len(is_p) > 0 and is_p.mean() > 0.4
            color = COLORS[1] if this_jet_is_p else COLORS[2]
            lw = 2 + 1 * (is_edj == this_jet_is_p)
            clu.axes[i].plot(lo - clu.central_longitude, la, color=color, lw=lw)
    # if key == "STJ":
    #     to_plot = da_z500_anom.sel(time=times)
    #     clu.add_contour(to_plot, levels=[-200, -100, 100, 200], colors="black", linewidths=2.)
    # if key == "STJ":
    #     to_plot = coarsen_da(da_tp.sel(time=times), 5)
    #     clu.add_contour(to_plot < 1e-5, levels=[0.5], colors=["green"], linewidths=3)
    #     break
    to_plot = coarsen_da(da_tp.sel(time=times), 3)
    clu.add_contour(to_plot * 1000, levels=[25], colors=["deepskyblue"], linewidths=1.4)
    if key == "EDJ":
        for (altitude, color), (orientation, hatch) in product(
            rwb_color.items(), rwb_hatching.items()
        ):
            to_plot = (
                das_rwb[f"{altitude}{orientation}PVS_new"].sel(time=times).any("lev")
            )
            for ax, ouais in zip(clu.axes, to_plot):
                cs = ax.pcolor(
                    to_plot.lon.values - clu.central_longitude,
                    to_plot.lat.values,
                    ouais.where(ouais),
                    hatch=hatch,
                    facecolor="none",
                    edgecolor=color,
                    hatch_linewidth=rwb_lw,
                    linewidth=0,
                    zorder=100,
                )

figure.savefig(f"{FIGURES}/jet_persistence/realspace_stuff/case_study_both.pdf")

In [ ]:
key = "EDJ"
which_spell = 1
spell = spells_list[key].filter(pl.col("spell") == which_spell).drop("jet ID")
spell = extend_spells(spell, time_before=datetime.timedelta(days=3))
spell = spell.filter((pl.col("relative_time") / datetime.timedelta(days=1)) % 3 == 0)
spell

## all of them

In [ ]:
from jetutils.data import compute_anomalies_pl, coarsen_da

n_t = 3
rwb_color = {
    "S": "mediumseagreen",
    "T": "orangered",
}
rwb_hatching = {
    "A": r"//",
    "C": r"\\",
}
rwb_lw = 2
cbar_kwargs = {"location": "bottom", "shrink": 0.7, "fraction": 0.1, "pad": 0.03}
plot_kwargs_all = {
    "STJ": {
        "da": da_T_anom,
        "cmap": colormaps.BlueWhiteOrangeRed,
        "levels": MaxNLocator(7).tick_values(-12, 12).tolist(),
        "cbar_kwargs": cbar_kwargs,
    },
    "EDJ": {
        "da": da_theta2pvu,
        "cmap": colormaps.bilbao_r,
        "levels": MaxNLocator(6).tick_values(300, 390).tolist(),
        "cbar_kwargs": cbar_kwargs,
    },
}
for spell_of in ["STJ", "EDJ"]:
    is_edj = spell_of == "EDJ"
    region = (-80, 40, 15, 80) if is_edj else (-80, 40, 0, 65)
    plot_kwargs = plot_kwargs_all[spell_of]
    spells = spells_list[spell_of].drop("jet ID")
    spells = extend_spells(spells, time_before=datetime.timedelta(days=3))
    spells = spells.filter((pl.col("relative_time") / datetime.timedelta(days=1)) % 3 == 0)
    da = plot_kwargs.pop("da")
    for n in range(30):
        clu = Clusterplot(n_t, 1, region, row_height=3.5)
        fig = clu.fig
        spell = spells.filter(pl.col("spell") == n)
        len_spell = spell["len"].first()
        times = spell["time"].unique().sort()[:n_t]
        titles = times.dt.to_string("%Y-%m-%dT%H:00").to_list()
        titles = [
            f"{ascii_lowercase[i]}) {title}" for i, title in enumerate(titles)
        ]
        to_plot = da.sel(time=times)
        clu.add_contourf(
            to_plot,
            titles=titles,
            cbar_label=PRETTIER_VARNAME.get(da.name, da.name),
            **plot_kwargs,
        )
        fig.suptitle(f"{spell_of} episode, " + str(len_spell)[:6])
        for i, t in enumerate(times):
            jets_ = jets.filter(pl.col("time") == t)
            for _, jet in jets_.group_by("jet ID"):
                lo, la, is_p = jet[["lon", "lat", "is_polar"]]
                this_jet_is_p = len(is_p) > 0 and (is_p < 0.5).mean() < 0.5
                this_jet_is_s = len(is_p) > 0 and (is_p < 0.5).mean() > 0.5
                color = COLORS[1] if this_jet_is_p else (COLORS[2] if this_jet_is_s else "gray")
                lw = 2 + 1 * (is_edj == this_jet_is_p)
                clu.axes[i].plot(lo - clu.central_longitude, la, color=color, lw=lw)
    
        to_plot = coarsen_da(da_tp_anom.sel(time=times), 3)
        color_tp = ["aquamarine"] if is_edj else ["cyan"]
        clu.add_contour(to_plot * 1000, levels=[15], colors=color_tp, linewidths=[1.4], linestyles=["solid"])
        if spell_of == "EDJ":
            for orientation, hatch in  rwb_hatching.items():
                to_plot = (
                    das_rwb[f"{orientation}PVO_new"].sel(time=times).any("lev")
                )
                for ax, ouais in zip(clu.axes, to_plot):
                    cs = ax.pcolor(
                        to_plot.lon.values - clu.central_longitude,
                        to_plot.lat.values,
                        ouais.where(ouais),
                        hatch=hatch,
                        facecolor="none",
                        edgecolor=rwb_color["S"],
                        hatch_linewidth=rwb_lw,
                        linewidth=0,
                        zorder=100,
                    )
        else:
            to_plot = da_z500_anom.sel(time=times)
            clu.add_contour(to_plot, levels=[-100, 100], colors="black", linewidths=2.)
        fig.savefig(f"{FIGURES}/jet_persistence/realspace_stuff/cs_{spell_of}_{n}.pdf")
        plt.close(fig)

    

In [ ]:
from scipy.stats import rankdata
da_tp_anom_ = da_tp_anom[:100, :20, :20]
da_tp_pvals = da_tp_anom_.copy(data=(rankdata(da_tp_anom_, axis=0) - 1) / len(da_tp_anom_.time))

In [ ]:
da_tp_pvals

In [ ]:
jets.filter(((pl.col("is_polar") < 0.5).n_unique() == 2).over("time", "jet ID")).group_by("time", "jet ID").agg(min=pl.col("is_polar").min(), max=pl.col("is_polar").max(), mean=pl.col("is_polar").mean(), prop_under=(pl.col("is_polar")<0.5).mean())

In [ ]:
to_save = {
    "t2m_anom": da_T_anom,
    "tp_anom": da_tp_anom,
    "tp": da_tp,
    "theta": da_theta2pvu,
    "z500_anom": da_z500_anom,
} | das_rwb
for name, da in to_save.items():
    for jet in ["STJ", "EDJ"]:
        spells = extend_spells(spells_list[jet], time_before=datetime.timedelta(days=4))
        spells = spells.select("spell", "relative_index", "time").unique(["spell", "relative_index"], maintain_order=True)
        to_save = []
        for _, spells_ in spells.group_by("spell", maintain_order=True):            
            times = polars_to_xarray(spells_, ["spell", "relative_index"])
            da_ = da.sel(time=times)
            to_save.append(da_)
        xr.concat(to_save, "spell").to_netcdf(basepath.joinpath(f"{name}_during_{jet}_spells.nc"))

In [ ]:
xr.open_dataarray(basepath.joinpath(f"{name}_during_{jet}_spells.nc"))

In [ ]:
times

In [ ]:
spells_list[jet]

In [ ]:
spells.unique(["spell", "relative_index"], maintain_order=True)

In [ ]:
polars_to_xarray(spells, ["spell", "relative_index"])

#